Forward and backward selection, principal component regression (PCR), partial least squares regression
(PLSR)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score

# Helper function to convert NOAA alphanumeric damage strings to numbers
def parse_damage(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip().upper()
    if val_str.endswith('K'):
        return float(val_str[:-1]) * 1_000
    elif val_str.endswith('M'):
        return float(val_str[:-1]) * 1_000_000
    elif val_str.endswith('B'):
        return float(val_str[:-1]) * 1_000_000_000
    try:
        return float(val_str)
    except ValueError:
        return 0.0

# 1. Load dataset
print("Loading capstone_2023_merged.csv...")
df = pd.read_csv("capstone_2023_merged.csv", low_memory=False)

# 2. Parse damage fields
if 'DAMAGE_PROPERTY' in df.columns:
    df['DAMAGE_PROPERTY_NUMERIC'] = df['DAMAGE_PROPERTY'].apply(parse_damage)

# 3. Feature and Target Selection
target_col = 'amountPaidOnBuildingClaim' if 'amountPaidOnBuildingClaim' in df.columns else 'amountPaidOnTotalClaim'
feature_cols = ['DAMAGE_PROPERTY_NUMERIC', 'primary_landcover_code', 'alternate_landcover_code']
feature_cols = [c for c in feature_cols if c in df.columns]

data = df[feature_cols + [target_col]].copy()
data[target_col] = pd.to_numeric(data[target_col], errors='coerce').fillna(0)
data = data.fillna(data.median(numeric_only=True))

X = data[feature_cols]
y_log = np.log1p(np.maximum(0, data[target_col]))

# 4. Generate Polynomial Matrix (9 Features)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

# 5. Train/Test Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(X_poly, y_log, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n--- Running Week 3 Models ---")

# --- 1. Forward Stepwise Selection ---
sfs_forward = SequentialFeatureSelector(LinearRegression(), n_features_to_select=4, direction='forward')
X_train_fwd = sfs_forward.fit_transform(X_train_scaled, y_train)
X_test_fwd = sfs_forward.transform(X_test_scaled)

model_fwd = LinearRegression()
model_fwd.fit(X_train_fwd, y_train)
r2_fwd = r2_score(y_test, model_fwd.predict(X_test_fwd))

# --- 2. Backward Stepwise Selection ---
sfs_backward = SequentialFeatureSelector(LinearRegression(), n_features_to_select=4, direction='backward')
X_train_bwd = sfs_backward.fit_transform(X_train_scaled, y_train)
X_test_bwd = sfs_backward.transform(X_test_scaled)

model_bwd = LinearRegression()
model_bwd.fit(X_train_bwd, y_train)
r2_bwd = r2_score(y_test, model_bwd.predict(X_test_bwd))

# --- 3. Principal Component Regression (PCR) ---
pca = PCA(n_components=3)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

model_pcr = LinearRegression()
model_pcr.fit(X_train_pca, y_train)
r2_pcr = r2_score(y_test, model_pcr.predict(X_test_pca))

# --- 4. Partial Least Squares Regression (PLSR) ---
pls = PLSRegression(n_components=3)
pls.fit(X_train_scaled, y_train)
r2_pls = r2_score(y_test, pls.predict(X_test_scaled))

# Results Summary
results_w3 = pd.DataFrame({
    'Technique': ['Forward Selection', 'Backward Selection', 'Principal Component Regression (PCR)', 'Partial Least Squares (PLSR)'],
    'Components/Features Selected': [4, 4, '3 Principal Components', '3 Latent Components'],
    'Test R²': [r2_fwd, r2_bwd, r2_pcr, r2_pls]
})

print("\n=== Week 3 Model Comparison Results ===")
print(results_w3.to_string(index=False))

Loading capstone_2023_merged.csv...

--- Running Week 3 Models ---

=== Week 3 Model Comparison Results ===
                           Technique Components/Features Selected   Test R²
                   Forward Selection                            4  0.003371
                  Backward Selection                            4  0.003300
Principal Component Regression (PCR)       3 Principal Components -0.000119
        Partial Least Squares (PLSR)          3 Latent Components  0.002613


In [2]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Results DataFrame
data = {
    'Technique': ['Forward Selection', 'Backward Selection', 'Partial Least Squares (PLSR)', 'Principal Component Regression (PCR)'],
    'Features/Components': ['4 Selected Features', '4 Selected Features', '3 Latent Components', '3 Principal Components'],
    'Test R²': [0.003371, 0.003300, 0.002613, -0.000119],
    'Primary Function': ['Greedy feature addition', 'Iterative feature pruning', 'Supervised dimension reduction', 'Unsupervised dimension reduction']
}

df = pd.DataFrame(data)

# 2. Render as Table Image
fig, ax = plt.subplots(figsize=(11, 3.5), dpi=300)
ax.axis('off')

table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    cellLoc='center',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

# Highlight header and style rows
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#40466e')
        cell.set_text_props(color='white', weight='bold')
    else:
        cell.set_facecolor('#f5f5f5' if row % 2 == 0 else 'white')

plt.savefig('week3_model_comparison.png', bbox_inches='tight', pad_inches=0.1)
plt.close()

print("Success! Saved Week 3 table image as 'week3_model_comparison.png'")

Success! Saved Week 3 table image as 'week3_model_comparison.png'
